# Localisation study — Part 1: matching figshare masks to the held-out test set

## Pre-specified analysis plan (written before any localisation result is computed)

**Purpose.** Test whether attribution maps (Grad-CAM++, IG, SmoothGrad, Equal Fusion, AGF) localise the tumour, using expert masks from the figshare brain-tumour dataset (Cheng et al.).

**Images.** Only figshare images identified in the Nickparvar **Testing/** set (never used for training, validation, or tuning). Figshare images found in **Training/** (train + validation) are excluded. No-tumour images have no masks and are excluded.

**Matching.** Perceptual hash (DCT, 64-bit) within the same class, confirmed by pixel correlation ≥ `CORR_MIN` after resizing the figshare image to the test image's size. Only one-to-one matches are kept.

**Masks.** Each figshare mask is resized to the raw test-image grid (nearest neighbour), then receives exactly the benchmark geometry: the black-border crop box computed from the test image (threshold 15, pad 4), then resize to 224×224.

**Metrics (Part 3).** Pointing Game (max-attribution pixel inside the mask), energy-in-mask (fraction of total attribution inside the mask), top-10% IoU with the mask. Reference baseline: a centred Gaussian map.

**Statistics (Part 3).** Paired Wilcoxon signed-rank (continuous metrics) and exact McNemar (Pointing Game), Holm-corrected within each family; median differences with 95% bootstrap CIs (1,000 resamples); rank-biserial effect sizes. Families: AGF vs each individual method; AGF vs Equal Fusion. Target class = clean prediction (as in the main study). Primary analysis: all matched test images. Sensitivity analyses: (i) correctly classified images only; (ii) images whose **patient** (figshare PID) does not appear in Training/.

In [ ]:
# ── Part 1 · Configuration and input discovery ─────────────────────────────
import os, glob, json
import numpy as np
from PIL import Image
from scipy.fft import dctn

SEED      = 42
CORR_MIN  = 0.90      # pixel-correlation threshold for a confirmed match (report distribution)
HAM_MAX   = 12        # max Hamming distance (of 64 bits) for a candidate
OUT       = "/kaggle/working/localisation"
os.makedirs(OUT, exist_ok=True)

_ROOTS = ["/kaggle/input/brain-tumor-mri-dataset",
          "/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset"]
DATASET_ROOT = next(p for p in _ROOTS if os.path.exists(p))
TEST_DIR, TRAIN_DIR = (os.path.join(DATASET_ROOT, s) for s in ("Testing", "Training"))

MAT_FILES = sorted(glob.glob("/kaggle/input/**/*.mat", recursive=True))
print("Nickparvar root:", DATASET_ROOT)
print("figshare .mat files found:", len(MAT_FILES))
assert len(MAT_FILES) > 3000, "Add the figshare brain-tumour dataset (.mat files with tumorMask) as Input"

TUMOUR_CLASSES = ["glioma", "meningioma", "pituitary"]
FIG_LABEL = {1: "meningioma", 2: "glioma", 3: "pituitary"}   # figshare label codes

In [ ]:
# ── Part 1 · Load figshare (MATLAB v7.3 / HDF5) ────────────────────────────
import h5py

def to_uint8(a):
    a = a.astype(np.float64)
    lo, hi = a.min(), a.max()
    return np.zeros(a.shape, np.uint8) if hi <= lo else ((a - lo) / (hi - lo) * 255).round().astype(np.uint8)

fig = []
for p in MAT_FILES:
    with h5py.File(p, "r") as f:
        cj  = f["cjdata"]
        img = np.array(cj["image"])
        msk = np.array(cj["tumorMask"]).astype(bool)
        lab = int(np.array(cj["label"]).squeeze())
        pid = "".join(chr(int(c)) for c in np.array(cj["PID"]).squeeze())
    fig.append({"file": os.path.basename(p), "img": to_uint8(img), "mask": msk,
                "cls": FIG_LABEL[lab], "pid": pid})
print(f"Loaded {len(fig)} figshare slices | patients: {len({r['pid'] for r in fig})}")
print({c: sum(r['cls'] == c for r in fig) for c in TUMOUR_CLASSES})

In [ ]:
# ── Part 1 · Perceptual hash (DCT) for both datasets ──────────────────────
def phash(gray_u8):
    small = np.asarray(Image.fromarray(gray_u8).resize((32, 32), Image.LANCZOS), np.float64)
    d = dctn(small, norm="ortho")[:8, :8].ravel()
    return (d > np.median(d[1:])).astype(np.uint8)

for r in fig:
    r["hash"]   = phash(r["img"])
    r["hash_T"] = phash(np.ascontiguousarray(r["img"].T))     # HDF5 may store MATLAB arrays transposed

def load_split(root, split):
    rows = []
    for c in TUMOUR_CLASSES:
        for p in sorted(glob.glob(os.path.join(root, c, "*"))):
            with Image.open(p) as im:
                g = np.asarray(im.convert("RGB").convert("L"))
            rows.append({"path": p, "split": split, "cls": c, "gray": g, "hash": phash(g)})
    return rows

test_rows  = load_split(TEST_DIR,  "test")
train_rows = load_split(TRAIN_DIR, "train")    # Training/ = train + validation
print(f"Nickparvar tumour images: test={len(test_rows)} train+val={len(train_rows)}")

In [ ]:
# ── Part 2 · Match Nickparvar images to figshare originals ─────────────────
def corr_at(fig_u8, target_gray):
    h, w = target_gray.shape
    a = np.asarray(Image.fromarray(fig_u8).resize((w, h), Image.BILINEAR), np.float64).ravel()
    b = target_gray.astype(np.float64).ravel()
    if a.std() == 0 or b.std() == 0:
        return -1.0
    return float(np.corrcoef(a, b)[0, 1])

fig_by_cls = {c: [i for i, r in enumerate(fig) if r["cls"] == c] for c in TUMOUR_CLASSES}
H  = {c: np.stack([fig[i]["hash"]   for i in fig_by_cls[c]]) for c in TUMOUR_CLASSES}
HT = {c: np.stack([fig[i]["hash_T"] for i in fig_by_cls[c]]) for c in TUMOUR_CLASSES}

def match(row):
    c = row["cls"]
    best = (None, -1.0, None)                      # (fig index, corr, transposed?)
    for hashes, transposed in ((H[c], False), (HT[c], True)):
        ham = (hashes != row["hash"]).sum(1)
        for k in np.where(ham <= HAM_MAX)[0]:
            fi  = fig_by_cls[c][k]
            src = fig[fi]["img"].T if transposed else fig[fi]["img"]
            r_  = corr_at(np.ascontiguousarray(src), row["gray"])
            if r_ > best[1]:
                best = (fi, r_, transposed)
    return best

all_corr = []
for rows in (test_rows, train_rows):
    for row in rows:
        fi, r_, tr = match(row)
        row.update(fig_idx=fi, corr=r_, transposed=tr)
        if fi is not None:
            all_corr.append(r_)

all_corr = np.array(all_corr)
print("Best-candidate correlation quantiles (5/25/50/75/95%):",
      np.round(np.quantile(all_corr, [.05, .25, .5, .75, .95]), 4))

In [ ]:
# ── Part 2 · Confirmed matches, one-to-one check, patient overlap ──────────
from collections import Counter

def confirmed(rows):
    return [r for r in rows if r["fig_idx"] is not None and r["corr"] >= CORR_MIN]

test_m, train_m = confirmed(test_rows), confirmed(train_rows)

# one-to-one: a figshare slice matched by more than one Nickparvar image signals duplicates
use = Counter(r["fig_idx"] for r in test_m + train_m)
dup = {fi for fi, n in use.items() if n > 1}
leak_slices = {r["fig_idx"] for r in test_m} & {r["fig_idx"] for r in train_m}
print(f"figshare slices matched more than once: {len(dup)} "
      f"(of which in BOTH test and train+val: {len(leak_slices)})")

test_final = [r for r in test_m if r["fig_idx"] not in dup]
train_pids = {fig[r["fig_idx"]]["pid"] for r in train_m}
for r in test_final:
    r["pid"] = fig[r["fig_idx"]]["pid"]
    r["pid_seen_in_train"] = r["pid"] in train_pids

print("\nConfirmed one-to-one test matches per class:",
      dict(Counter(r["cls"] for r in test_final)), "| total:", len(test_final))
print("Test tumour images without a figshare match (SARTAJ or unmatched):",
      len(test_rows) - len(test_m))
print("Matched test images whose PATIENT also appears in train+val:",
      sum(r["pid_seen_in_train"] for r in test_final), "/", len(test_final))

In [ ]:
# ── Part 3 · Align each mask to the benchmark geometry ────────────────────
# Replicates RemoveBlackBorder exactly (threshold 15 on the 'L' image, pad 4,
# half-open crop box, MIN_SIDE 8), then the direct 224x224 resize.
def border_box(rgb_img):
    arr  = np.array(rgb_img.convert("L"))
    mask = arr > 15
    rows, cols = np.any(mask, axis=1), np.any(mask, axis=0)
    h, w = arr.shape
    if not rows.any() or not cols.any():
        return (0, 0, w, h)
    r0, r1 = np.where(rows)[0][[0, -1]]
    c0, c1 = np.where(cols)[0][[0, -1]]
    r0, r1 = max(0, r0 - 4), min(h - 1, r1 + 4)
    c0, c1 = max(0, c0 - 4), min(w - 1, c1 + 4)
    box = (int(c0), int(r0), int(c1) + 1, int(r1) + 1)
    if box[2] - box[0] < 8 or box[3] - box[1] < 8:
        return (0, 0, w, h)
    return box

def aligned_mask(row):
    fr   = fig[row["fig_idx"]]
    m    = fr["mask"].T if row["transposed"] else fr["mask"]
    h, w = row["gray"].shape
    m    = Image.fromarray(np.ascontiguousarray(m).astype(np.uint8) * 255).resize((w, h), Image.NEAREST)
    with Image.open(row["path"]) as im:
        box = border_box(im.convert("RGB"))
    m = m.crop(box).resize((224, 224), Image.NEAREST)
    return np.asarray(m) > 127

masks = {}
for r in test_final:
    m = aligned_mask(r)
    if m.sum() > 0:
        masks[r["path"]] = m

meta = [{"path": r["path"], "cls": r["cls"], "fig_file": fig[r["fig_idx"]]["file"],
         "pid": r["pid"], "pid_seen_in_train": bool(r["pid_seen_in_train"]),
         "corr": round(r["corr"], 4), "transposed": bool(r["transposed"]),
         "mask_frac": float(masks[r["path"]].mean())}
        for r in test_final if r["path"] in masks]
np.savez_compressed(os.path.join(OUT, "test_masks_224.npz"),
                    paths=np.array([m["path"] for m in meta]),
                    masks=np.stack([masks[m["path"]] for m in meta]))
with open(os.path.join(OUT, "test_mask_meta.json"), "w") as f:
    json.dump(meta, f, indent=1)
print(f"Saved {len(meta)} aligned masks -> {OUT}")
print("Median tumour area (fraction of 224x224):", round(float(np.median([m['mask_frac'] for m in meta])), 4))

In [ ]:
# ── Part 3 · Visual alignment check ──
import matplotlib.pyplot as plt
from skimage import measure
import torchvision.transforms as T

rng  = np.random.default_rng(SEED)
pick = [meta[i] for i in rng.choice(len(meta), size=min(10, len(meta)), replace=False)]
resize224 = T.Resize((224, 224))

fig_, axes = plt.subplots(2, 5, figsize=(16, 7))
for ax, m in zip(axes.ravel(), pick):
    with Image.open(m["path"]) as im:
        im = im.convert("RGB")
        shown = resize224(im.crop(border_box(im)))
    ax.imshow(shown, cmap="gray")
    for c in measure.find_contours(masks[m["path"]].astype(float), 0.5):
        ax.plot(c[:, 1], c[:, 0], color="red", lw=1.5)
    ax.set_title(f"{m['cls']} | r={m['corr']:.3f}", fontsize=9)
    ax.axis("off")
plt.suptitle("Alignment check: figshare tumour mask (red) on the preprocessed test image")
plt.tight_layout()
plt.savefig(os.path.join(OUT, "alignment_check.png"), dpi=150)
plt.show()